In [ ]:
"""
KG-TRACEABLE QA/MCQ PIPELINE (NebulaGraph + FAISS + CrossEncoder + optional BM25)

1) TWO MODES:
   - mode="mcq": option-aware evidence pruning (uses options ONLY to rank/keep evidence that separates options).
   - mode="chat": option-agnostic pruning (question-only), suitable for chatbot usage.
   No "option coverage" enforcement. We do NOT try to get evidence per option.

2) BM25 PATCH (generic, not question-specific):
   - Adds lightweight BM25 rescoring for:
     (a) semantic-name candidates (to suppress generic "lab test"/noise)
     (b) KG definition texts
     (c) dense FAISS definition texts
   BM25 is computed ONLY over the candidate pool (no global index), so it works for all questions.

3) CUDA ALWAYS (NO env vars):
   - SentenceTransformer(..., device="cuda")
   - CrossEncoder(..., device="cuda")
   If CUDA is unavailable, it will throw (which is what you want; no silent CPU fallback).

4) Keeps your exact configs (hard-coded API key etc.)
5) "LAST DEF QUERY" printing: prints ONLY the last FETCH PROP ON Definition ... query.

IMPORTANT NOTE (re: fairness):
- mode="mcq" uses options ONLY in the pruning stage; retrieval stays single joint query (question+options).
- mode="chat" never uses options; it’s the chatbot-safe mode.
"""

import json
import re
import ast
import math
import logging
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field

from nebula3.Config import Config
from nebula3.gclient.net import ConnectionPool
import httpx

# ----------------------------
# LOGGING
# ----------------------------
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("kg_mcq")

# ----------------------------
# CONFIG (your exact configs)
# ----------------------------
EMBED_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"

DEF_INDEX_FILE = "def_faiss.index"
DEF_META_FILE  = "def_meta.filtered.json"

SEM_INDEX_FILE = "semantic_faiss.index"
SEM_META_JSONL = "semantic_nodes.filtered.jsonl"

NEBULA_HOST = "127.0.0.1"
NEBULA_PORT = 9669
NEBULA_USER = "root"
NEBULA_PASSWORD = "nebula"
NEBULA_SPACE = "petagraph"

OPENAI_BASE_URL = "https://ollama.zib.de/api"
API_KEY = "sk-50837c3046664c90bc4367c9d5ebff3f"
MODEL_NAME = "gemma2:27b"

CROSS_ENCODER_MODEL = "ncbi/MedCPT-Cross-Encoder"

DATA_DIR = "."  # NO env vars

# ----------------------------
# UTILS
# ----------------------------
def safe_literal_eval_dict(x: Any) -> Dict[str, str]:
    if isinstance(x, dict):
        return {str(k): str(v) for k, v in x.items()}
    if isinstance(x, str):
        s = x.strip()
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return {str(k): str(v) for k, v in obj.items()}
        except Exception:
            pass
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, dict):
                return {str(k): str(v) for k, v in obj.items()}
        except Exception:
            pass
    raise ValueError(f"Could not parse options into a dict. Got: {type(x)}")

def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def build_joint_query(question: str, options: Dict[str, str]) -> str:
    opt_str = " ".join([f"({k}) {v}" for k, v in sorted(options.items(), key=lambda kv: kv[0])])
    return normalize_text(f"Question: {question}\nOptions: {opt_str}")

def build_question_only_query(question: str) -> str:
    return normalize_text(f"Question: {question}")

EID_RE = re.compile(r"\[(E\d+)\]")

def extract_eids_from_text(*texts: str) -> List[str]:
    """Single canonical helper (no duplicate versions)."""
    found: List[str] = []
    for t in texts:
        if t:
            found.extend(EID_RE.findall(t))
    # dedupe preserving order
    out = []
    seen = set()
    for e in found:
        if e not in seen:
            seen.add(e)
            out.append(e)
    return out

def sigmoid(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))

def minmax01(x: np.ndarray) -> np.ndarray:
    if x.size == 0:
        return x
    lo = float(np.min(x))
    hi = float(np.max(x))
    if hi <= lo + 1e-12:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - lo) / (hi - lo)).astype(np.float32)

# ----------------------------
# TOKENIZATION + BM25 (lightweight, candidate-set BM25)
# ----------------------------
_TOKEN_RE = re.compile(r"[A-Za-z0-9]+")  # simple, fast

def bm25_tokenize(text: str) -> List[str]:
    return [t.lower() for t in _TOKEN_RE.findall(text or "")]

def bm25_scores(query: str, docs: List[str], k1: float = 1.6, b: float = 0.75) -> np.ndarray:
    """
    Candidate-set BM25:
    - IDF computed over provided docs only
    - Good for suppressing generic noise without adding a new retrieval system
    """
    if not docs:
        return np.zeros((0,), dtype=np.float32)

    q_tokens = bm25_tokenize(query)
    if not q_tokens:
        return np.zeros((len(docs),), dtype=np.float32)

    docs_tokens = [bm25_tokenize(d) for d in docs]
    N = len(docs_tokens)
    dl = np.array([len(toks) for toks in docs_tokens], dtype=np.float32)
    avgdl = float(np.mean(dl)) if N > 0 else 1.0
    avgdl = max(avgdl, 1.0)

    # df
    df: Dict[str, int] = {}
    for toks in docs_tokens:
        for w in set(toks):
            df[w] = df.get(w, 0) + 1

    # idf (BM25+1 style)
    def idf(w: str) -> float:
        n = df.get(w, 0)
        return math.log(1.0 + (N - n + 0.5) / (n + 0.5))

    # tf per doc
    scores = np.zeros((N,), dtype=np.float32)
    for i, toks in enumerate(docs_tokens):
        if not toks:
            continue
        tf: Dict[str, int] = {}
        for w in toks:
            tf[w] = tf.get(w, 0) + 1

        s = 0.0
        denom_base = k1 * (1.0 - b + b * (dl[i] / avgdl))
        for w in q_tokens:
            f = tf.get(w, 0)
            if f <= 0:
                continue
            numer = f * (k1 + 1.0)
            denom = f + denom_base
            s += idf(w) * (numer / max(denom, 1e-12))
        scores[i] = float(s)

    return scores

# ----------------------------
# MMR selection (diversity over semantic-name embeddings)
# ----------------------------
def mmr_select(
    query_emb: np.ndarray,
    cand_embs: np.ndarray,
    cand_scores: np.ndarray,
    k: int = 60,
    lambda_mult: float = 0.7,
) -> List[int]:
    if cand_embs.shape[0] == 0:
        return []
    k = min(k, cand_embs.shape[0])

    q = query_emb.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-12)

    E = cand_embs.astype(np.float32)
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)

    rel = (E @ q).reshape(-1)  # cosine
    # cand_scores: FAISS distances (higher is "more similar" for IP indices; for L2 indices it's opposite)
    # We don't assume index type; just normalize and mix lightly.
    s_norm = cand_scores.astype(np.float32)
    s_norm = (s_norm - np.min(s_norm)) / (np.max(s_norm) - np.min(s_norm) + 1e-12)
    rel2 = 0.7 * rel + 0.3 * s_norm

    selected: List[int] = []
    remaining = list(range(E.shape[0]))

    while len(selected) < k and remaining:
        if not selected:
            best = max(remaining, key=lambda i: float(rel2[i]))
            selected.append(best)
            remaining.remove(best)
            continue

        S = E[selected]
        sims = E[remaining] @ S.T
        max_sim = np.max(sims, axis=1)

        mmr = lambda_mult * rel2[remaining] - (1.0 - lambda_mult) * max_sim
        best_pos = int(np.argmax(mmr))
        best = remaining[best_pos]
        selected.append(best)
        remaining.remove(best)

    return selected

# ----------------------------
# Evidence structures
# ----------------------------
@dataclass
class EvidenceNode:
    eid: str
    route: str
    score01: float
    text: str
    meta: Dict[str, Any]
    trace: Optional[Dict[str, Any]] = None

# ----------------------------
# Output schema
# ----------------------------
class MCQZipAnswer(BaseModel):
    cop_index: str = Field(..., description="Correct option index as string, or '-1'")
    answer: str = Field(..., description="Option text if cop_index != -1 else ''")
    why_correct: str = Field(..., description="Short explanation with evidence citations")
    why_others_incorrect: Dict[str, str] = Field(..., description="Per-option why incorrect")
    evidence_used: List[str] = Field(default_factory=list, description="Evidence IDs actually used (E#)")
    evidence: Dict[str, Any] = Field(
        default_factory=dict,
        description="Map of evidence_id -> {text, route, score, ids, trace(optional)}; ONLY for evidence_used"
    )

def build_evidence_payload(nodes: List[EvidenceNode], used_eids: List[str]) -> Dict[str, Any]:
    by_eid = {n.eid: n for n in nodes}
    payload: Dict[str, Any] = {}
    for eid in used_eids:
        n = by_eid.get(eid)
        if not n:
            continue
        item = {
            "text": n.text,
            "route": n.route,
            "score": n.score01,
        }
        # include common IDs from meta if present
        ids = {}
        for k in ("sui", "name", "concept_vid", "def_vid", "ATUI", "SAB"):
            v = n.meta.get(k)
            if v not in (None, "", "None"):
                ids[k] = v
        if ids:
            item["ids"] = ids
        if n.trace:
            item["trace"] = n.trace
        payload[eid] = item
    return payload

# ----------------------------
# Nebula helpers
# ----------------------------
def _to_str(v: Any) -> str:
    if v is None:
        return ""
    try:
        return str(v).strip('"')
    except Exception:
        return ""

def _row_get(row: Any, idx: int) -> Any:
    # robust for nebula returning row-values as list or Row object
    if row is None:
        return None
    if isinstance(row, (list, tuple)):
        return row[idx] if idx < len(row) else None
    if hasattr(row, "values"):
        try:
            return row.values[idx]
        except Exception:
            return None
    try:
        return row[idx]
    except Exception:
        return None

class NebulaClient:
    def __init__(self, host, port, user, password, space):
        cfg = Config()
        cfg.max_connection_pool_size = 10
        self.pool = ConnectionPool()
        ok = self.pool.init([(host, port)], cfg)
        if not ok:
            raise RuntimeError("Failed to init Nebula connection pool")
        self.sess = self.pool.get_session(user, password, True)
        self.sess.execute(f"USE {space}")

    def close(self):
        try:
            self.sess.release()
        except Exception:
            pass
        try:
            self.pool.close()
        except Exception:
            pass

    def _exec(self, ngql: str):
        res = self.sess.execute(ngql)
        if not res.is_succeeded():
            raise RuntimeError(f"Nebula query failed: {res.error_msg()}\nNGQL:\n{ngql}")
        return res

    def lookup_semantic_vids(self, suis: List[str], limit_each: int = 1) -> Dict[str, str]:
        out = {}
        for s in suis:
            ngql = f'''
LOOKUP ON Semantic WHERE Semantic.SUI == "{s}"
YIELD id(vertex) AS vid
| LIMIT {int(limit_each)};
'''.strip()
            res = self._exec(ngql)
            if res.row_size() > 0:
                row = res.row_values(0)
                v = _row_get(row, 0)
                out[s] = _to_str(v)
        return out

    def sty_reverse_concepts(self, sem_vids: List[str], limit_each: int = 200) -> Dict[str, List[str]]:
        out: Dict[str, List[str]] = {}
        for sv in sem_vids:
            ngql = f'''
GO FROM "{sv}" OVER STY REVERSELY
YIELD DISTINCT id($$) AS concept_vid
| LIMIT {int(limit_each)};
'''.strip()
            res = self._exec(ngql)
            concepts = []
            for i in range(res.row_size()):
                row = res.row_values(i)
                c = _row_get(row, 0)
                concepts.append(_to_str(c))
            out[sv] = concepts
        return out

    def def_pairs_for_concepts(self, concept_vids: List[str], limit_total: int = 6000) -> Tuple[List[Tuple[str, str]], str]:
        if not concept_vids:
            return [], ""
        quoted = ", ".join([f"\"{c}\"" for c in concept_vids])
        def_edge_query = f'''
GO FROM {quoted} OVER DEF
YIELD DISTINCT id($^) AS concept_vid, id($$) AS def_vid
| LIMIT {int(limit_total)};
'''.strip()

        res = self._exec(def_edge_query)
        pairs = []
        for i in range(res.row_size()):
            row = res.row_values(i)
            cvid = _to_str(_row_get(row, 0))
            dvid = _to_str(_row_get(row, 1))
            if cvid and dvid:
                pairs.append((cvid, dvid))

        return pairs, def_edge_query

    def fetch_def_props(self, def_vids: List[str]) -> Tuple[Dict[str, Dict[str, Any]], str]:
        if not def_vids:
            return {}, ""
        chunk_size = 200
        out: Dict[str, Dict[str, Any]] = {}
        last_fetch_query = ""

        for i in range(0, len(def_vids), chunk_size):
            chunk = def_vids[i:i + chunk_size]
            q = ", ".join([f"\"{d}\"" for d in chunk])
            ngql = f'''
FETCH PROP ON Definition {q}
YIELD id(vertex) AS def_vid, Definition.ATUI AS atui, Definition.DEF AS def_text, Definition.SAB AS sab;
'''.strip()
            last_fetch_query = ngql

            res = self._exec(ngql)
            for r in range(res.row_size()):
                row = res.row_values(r)
                def_vid = _to_str(_row_get(row, 0))
                atui    = _to_str(_row_get(row, 1))
                def_txt = _to_str(_row_get(row, 2))
                sab     = _to_str(_row_get(row, 3))
                if def_vid:
                    out[def_vid] = {"def_vid": def_vid, "ATUI": atui, "SAB": sab, "DEF": def_txt}

        return out, last_fetch_query

# ----------------------------
# FAISS wrapper (JSONL meta supported)
# ----------------------------
def read_json_or_jsonl(path: str) -> Any:
    p = str(path)
    if p.endswith(".jsonl"):
        out = []
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                out.append(json.loads(line))
        return out
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

class FaissIndex:
    def __init__(self, index_path: str, meta_path: str):
        self.index = faiss.read_index(index_path)
        self.meta = read_json_or_jsonl(meta_path)
        if not isinstance(self.meta, list):
            raise RuntimeError(f"Meta must be list. Got: {type(self.meta)}")
        if self.index.ntotal != len(self.meta):
            raise RuntimeError(
                f"FAISS ntotal={self.index.ntotal} != meta len={len(self.meta)}\n"
                f"Index: {index_path}\nMeta : {meta_path}"
            )

    def search(self, emb: np.ndarray, topk: int) -> Tuple[np.ndarray, np.ndarray]:
        D, I = self.index.search(emb.astype(np.float32), topk)
        return D[0], I[0]

# ----------------------------
# Ollama OpenAI-like client
# ----------------------------
class OpenAILikeOllama:
    def __init__(self, base_url: str, model: str, api_key: str, timeout_s: int = 120):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key
        self.client = httpx.Client(timeout=timeout_s)

    def chat(self, messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
        url = f"{self.base_url}/chat/completions"
        payload = {"model": self.model, "messages": messages, "temperature": float(temperature)}
        headers = {"Authorization": f"Bearer {self.api_key}"}
        r = self.client.post(url, json=payload, headers=headers)
        r.raise_for_status()
        data = r.json()
        return data["choices"][0]["message"]["content"]

# ----------------------------
# JSON extraction + sanitize
# ----------------------------
def extract_first_json_obj(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    blob = m.group(0).strip()
    try:
        return json.loads(blob)
    except Exception:
        pass
    try:
        obj = ast.literal_eval(blob)
        if isinstance(obj, dict):
            return obj
    except Exception:
        return None
    return None

def sanitize_zip_answer(
    obj: Dict[str, Any],
    options: Dict[str, str],
    allowed_eids: List[str],
    fallback_reason: str,
    fallback_evidence: Optional[List[str]],
) -> MCQZipAnswer:
    option_keys = sorted(options.keys(), key=lambda x: int(x) if x.isdigit() else x)
    option_key_set = set(option_keys)
    allowed_set = set(allowed_eids)

    cop = str(obj.get("cop_index", "-1"))
    if cop not in option_key_set and cop != "-1":
        cop = "-1"

    answer = "" if cop == "-1" else options.get(cop, "")

    why_correct = str(obj.get("why_correct", "") or "").strip()

    why_others = obj.get("why_others_incorrect", {})
    if not isinstance(why_others, dict):
        why_others = {}
    why_others = {str(k): str(v) for k, v in why_others.items()}

    # in MCQ output, keep only "other" keys; for abstain keep all
    if cop == "-1":
        keys_to_include = option_keys[:]
    else:
        keys_to_include = [k for k in option_keys if k != cop]

    ordered_why_others: Dict[str, str] = {}
    for k in keys_to_include:
        ordered_why_others[k] = (why_others.get(k, "") or "").strip()

    # evidence_used must match citations
    cited = extract_eids_from_text(why_correct, *list(ordered_why_others.values()))
    cited = [e for e in cited if e in allowed_set]
    evidence_used = cited[:]

    if not evidence_used and fallback_evidence:
        evidence_used = [fallback_evidence[0]]

    # ensure why_correct exists & cites something
    if not why_correct:
        if evidence_used:
            why_correct = f"{fallback_reason} [{evidence_used[0]}]"
        else:
            why_correct = fallback_reason
    else:
        if not extract_eids_from_text(why_correct) and evidence_used:
            why_correct = f"{why_correct} [{evidence_used[0]}]"

    if cop == "-1":
        answer = ""

    return MCQZipAnswer(
        cop_index=cop,
        answer=answer,
        why_correct=why_correct,
        why_others_incorrect=ordered_why_others,
        evidence_used=evidence_used,
        evidence={}
    )

# ----------------------------
# GLOBAL EVIDENCE PRUNING (two modes)
# ----------------------------
def node_content_for_scoring(n: EvidenceNode) -> str:
    # what BM25 sees for evidence
    return normalize_text(n.text)

def compute_node_bm25_scores(query: str, nodes: List[EvidenceNode]) -> np.ndarray:
    docs = [node_content_for_scoring(n) for n in nodes]
    s = bm25_scores(query, docs)
    return s

def mcq_discriminative_margin(question: str, options: Dict[str, str], nodes: List[EvidenceNode]) -> np.ndarray:
    """
    Option-aware but generic:
    For each evidence node, compute BM25 score against each (question + option_text),
    then take margin = best - second_best. Big margin => evidence distinguishes options.
    """
    if not nodes or not options:
        return np.zeros((len(nodes),), dtype=np.float32)

    option_keys = sorted(options.keys(), key=lambda x: int(x) if x.isdigit() else x)
    # build per-option queries
    queries = [normalize_text(f"{question} {options[k]}") for k in option_keys]
    docs = [node_content_for_scoring(n) for n in nodes]

    per_opt = []
    for q in queries:
        per_opt.append(bm25_scores(q, docs))
    M = np.stack(per_opt, axis=1)  # [N_nodes, N_opts]

    # margin best - 2nd best
    # sort descending along opts
    sorted_vals = np.sort(M, axis=1)[:, ::-1]
    best = sorted_vals[:, 0]
    second = sorted_vals[:, 1] if sorted_vals.shape[1] >= 2 else np.zeros_like(best)
    margin = best - second
    return margin.astype(np.float32)

def prune_nodes(
    *,
    mode: str,
    question: str,
    options: Dict[str, str],
    nodes: List[EvidenceNode],
    keep_k: int,
    min_keep: int = 4,
    # weights
    w_base: float = 0.55,
    w_bm25_q: float = 0.30,
    w_margin: float = 0.15,
) -> List[EvidenceNode]:
    """
    mode:
      - "chat": score by base score01 + BM25(question-only)
      - "mcq" : score by base score01 + BM25(joint query) + discriminative margin
    """
    if not nodes:
        return []

    mode = (mode or "mcq").lower().strip()
    keep_k = max(int(keep_k), int(min_keep))

    base = np.array([float(n.score01) for n in nodes], dtype=np.float32)
    base = minmax01(base)

    if mode == "chat":
        q = build_question_only_query(question)
        bm25_q = compute_node_bm25_scores(q, nodes)
        bm25_q = minmax01(bm25_q)

        final = w_base * base + w_bm25_q * bm25_q
    else:
        # mcq
        joint = build_joint_query(question, options) if options else build_question_only_query(question)
        bm25_q = compute_node_bm25_scores(joint, nodes)
        bm25_q = minmax01(bm25_q)

        margin = mcq_discriminative_margin(question, options, nodes) if options else np.zeros_like(bm25_q)
        margin = minmax01(margin)

        final = w_base * base + w_bm25_q * bm25_q + w_margin * margin

    order = np.argsort(-final)
    chosen = order[: min(keep_k, len(nodes))].tolist()
    chosen = chosen[: max(min_keep, len(chosen))]

    out = []
    for i in chosen:
        n = nodes[int(i)]
        # overwrite displayed score01 with the "final" prune score for debugging transparency
        out.append(EvidenceNode(
            eid=n.eid,
            route=n.route,
            score01=float(final[int(i)]),
            text=n.text,
            meta=n.meta,
            trace=n.trace
        ))
    return out

# ----------------------------
# PIPELINE
# ----------------------------
class KGTraceableMCQPipeline:
    def __init__(
        self,
        semantic_faiss: FaissIndex,
        def_faiss: FaissIndex,
        embedder: SentenceTransformer,
        cross_encoder: CrossEncoder,
        nebula: NebulaClient,
        llm: OpenAILikeOllama,
        *,
        semantic_topk_dense: int = 220,
        seed_k_mmr: int = 60,
        mmr_lambda: float = 0.7,
        dense_def_topk: int = 80,
        dense_defs_keep: int = 20,
        concept_limit_each_sem: int = 200,
        def_pairs_limit_total: int = 6000,
        kg_defs_keep: int = 25,
        semantic_name_keep: int = 10,
        temperature: float = 0.0,
        # Policy
        require_kg_def_if_available: bool = True,
        kg_min_sigmoid_to_enforce: float = 0.70,  # absolute threshold, no minmax inflation
        retry_strict_once: bool = True,
        # Evidence prune
        prune_keep_k: int = 32,
        prune_min_keep: int = 4,
    ):
        self.semantic_faiss = semantic_faiss
        self.def_faiss = def_faiss
        self.embedder = embedder
        self.cross_encoder = cross_encoder
        self.nebula = nebula
        self.llm = llm

        self.semantic_topk_dense = semantic_topk_dense
        self.seed_k_mmr = seed_k_mmr
        self.mmr_lambda = mmr_lambda
        self.dense_def_topk = dense_def_topk
        self.dense_defs_keep = dense_defs_keep

        self.concept_limit_each_sem = concept_limit_each_sem
        self.def_pairs_limit_total = def_pairs_limit_total
        self.kg_defs_keep = kg_defs_keep
        self.semantic_name_keep = semantic_name_keep

        self.temperature = temperature
        self.require_kg_def_if_available = require_kg_def_if_available
        self.kg_min_sigmoid_to_enforce = kg_min_sigmoid_to_enforce
        self.retry_strict_once = retry_strict_once

        self.prune_keep_k = prune_keep_k
        self.prune_min_keep = prune_min_keep

    # ---- semantic seeds (MMR) ----
    def retrieve_seeds(self, joint_query: str) -> Dict[str, Any]:
        q_emb = self.embedder.encode([joint_query], normalize_embeddings=True).astype(np.float32)
        D, I = self.semantic_faiss.search(q_emb, self.semantic_topk_dense)

        cand_meta = [self.semantic_faiss.meta[int(i)] for i in I if int(i) >= 0]

        def get_sui(m):  return m.get("sui") or m.get("SUI")
        def get_name(m): return m.get("name") or m.get("NAME")

        cand_sui = [get_sui(m) for m in cand_meta]
        cand_name = [get_name(m) for m in cand_meta]

        # MMR over name embeddings
        name_embs = self.embedder.encode([normalize_text(n) for n in cand_name], normalize_embeddings=True).astype(np.float32)

        mmr_idx = mmr_select(
            query_emb=q_emb[0],
            cand_embs=name_embs,
            cand_scores=np.array(D, dtype=np.float32),
            k=self.seed_k_mmr,
            lambda_mult=self.mmr_lambda,
        )

        seeds_mmr = [{"sui": str(cand_sui[idx]), "name": str(cand_name[idx]), "dense_score": float(D[idx])} for idx in mmr_idx]

        # Semantic-name evidence candidates (BM25-filtered later in pruning stage)
        top_dense = [{"sui": str(cand_sui[idx]), "name": str(cand_name[idx]), "dense_score": float(D[idx])}
                     for idx in range(min(self.semantic_name_keep * 6, len(cand_sui)))]  # take wider pool, prune later

        return {"joint_query": joint_query, "q_emb": q_emb, "top_dense": top_dense, "seeds_mmr": seeds_mmr}

    # ---- dense def fallback ----
    def dense_def_candidates(self, joint_query: str) -> List[Dict[str, Any]]:
        q_emb = self.embedder.encode([joint_query], normalize_embeddings=True).astype(np.float32)
        D, I = self.def_faiss.search(q_emb, self.dense_def_topk)

        cands = []
        for dist, idx in zip(D.tolist(), I.tolist()):
            if idx < 0:
                continue
            m = self.def_faiss.meta[int(idx)]
            atui = m.get("ATUI") or m.get("atui") or ""
            sab  = m.get("SAB") or m.get("sab") or ""
            dtx  = m.get("DEF") or m.get("def") or m.get("def_text") or ""
            if not dtx:
                continue
            cands.append({"ATUI": str(atui), "SAB": str(sab), "DEF": str(dtx), "dense_score": float(dist)})
        return cands

    def rerank_defs_with_cross_and_bm25(
        self,
        query: str,
        defs: List[str],
        *,
        w_ce: float = 0.75,
        w_bm25: float = 0.25,
    ) -> np.ndarray:
        """
        Returns score01 in [0,1] per def text
        - CrossEncoder sigmoid score (absolute) + BM25(candidate-set) normalized
        """
        if not defs:
            return np.zeros((0,), dtype=np.float32)

        pairs = [(query, normalize_text(d)) for d in defs]
        raw = np.array(self.cross_encoder.predict(pairs), dtype=np.float32)
        ce01 = sigmoid(raw)

        bm = bm25_scores(query, [normalize_text(d) for d in defs])
        bm01 = minmax01(bm)

        score = (w_ce * ce01 + w_bm25 * bm01).astype(np.float32)
        return score

    def rerank_dense_defs(self, joint_query: str, dense_defs: List[Dict[str, Any]]) -> List[Tuple[int, float]]:
        if not dense_defs:
            return []
        texts = [d["DEF"] for d in dense_defs]
        score01 = self.rerank_defs_with_cross_and_bm25(joint_query, texts, w_ce=0.75, w_bm25=0.25)
        ranked = sorted(list(enumerate(score01.tolist())), key=lambda x: x[1], reverse=True)
        return ranked

    # ---- KG defs from seeds ----
    def kg_defs_from_seeds(self, seeds: List[Dict[str, Any]]) -> Dict[str, Any]:
        seed_suis = [s["sui"] for s in seeds if s.get("sui") and s.get("sui") != "None"]

        lookup = self.nebula.lookup_semantic_vids(seed_suis, limit_each=1)
        sem_vids = list(lookup.values())

        sem_to_concepts = self.nebula.sty_reverse_concepts(sem_vids, limit_each=self.concept_limit_each_sem)

        all_concepts = []
        for cs in sem_to_concepts.values():
            all_concepts.extend(cs)
        all_concepts = list(dict.fromkeys(all_concepts))

        pairs, _def_edge_query = self.nebula.def_pairs_for_concepts(all_concepts, limit_total=self.def_pairs_limit_total)
        def_vids = list(dict.fromkeys([d for (_, d) in pairs]))

        def_props, last_fetch_def_query = self.nebula.fetch_def_props(def_vids)

        semvid_to_seed = {lookup[sui]: sui for sui in lookup if lookup.get(sui)}
        concept_to_semvid = {}
        for sv, cs in sem_to_concepts.items():
            for c in cs:
                concept_to_semvid[c] = sv

        def_trace = {}
        for cvid, dvid in pairs:
            sv = concept_to_semvid.get(cvid)
            seed_sui = semvid_to_seed.get(sv)
            if dvid not in def_trace:
                def_trace[dvid] = {"seed_sui": seed_sui, "sem_vid": sv, "concept_vid": cvid, "def_vid": dvid}

        return {
            "seed_suis": seed_suis,
            "semantic_vids": sem_vids,
            "concept_vids": all_concepts,
            "def_pairs": pairs,
            "def_props": def_props,
            "def_trace": def_trace,
            "last_fetch_def_query": last_fetch_def_query,
        }

    def rerank_kg_defs(self, joint_query: str, kg_def_props: Dict[str, Dict[str, Any]]) -> List[Tuple[str, float]]:
        items = [(dvid, normalize_text(p.get("DEF", ""))) for dvid, p in kg_def_props.items()]
        items = [(dvid, txt) for dvid, txt in items if txt]
        if not items:
            return []
        texts = [txt for _, txt in items]
        score01 = self.rerank_defs_with_cross_and_bm25(joint_query, texts, w_ce=0.75, w_bm25=0.25)
        ranked = sorted(zip([dvid for dvid, _ in items], score01.tolist()), key=lambda x: x[1], reverse=True)
        return ranked

    # ---- evidence nodes ----
    def build_evidence_nodes(
        self,
        retrieval: Dict[str, Any],
        kg: Dict[str, Any],
        kg_ranked: List[Tuple[str, float]],
        dense_defs: List[Dict[str, Any]],
        dense_ranked: List[Tuple[int, float]],
        *,
        question: str,
        options: Dict[str, str],
        mode: str,
    ) -> List[EvidenceNode]:
        nodes: List[EvidenceNode] = []
        eid_counter = 1

        # 1) KG defs first
        for def_vid, s01 in kg_ranked[: self.kg_defs_keep]:
            props = kg["def_props"].get(def_vid, {})
            def_text = props.get("DEF", "")
            atui = props.get("ATUI", "")
            sab = props.get("SAB", "")

            core = kg["def_trace"].get(def_vid, {})
            trace_obj = {
                "route": "kg_sui_concept_def",
                "seed_sui": core.get("seed_sui"),
                "sem_vid": core.get("sem_vid"),
                "concept_vid": core.get("concept_vid"),
                "def_vid": def_vid,
                "atui": atui,
                "sab": sab,
            }

            eid = f"E{eid_counter}"; eid_counter += 1
            nodes.append(EvidenceNode(
                eid=eid,
                route="kg_sui_concept_def",
                score01=float(s01),
                text=f'Definition: "{normalize_text(def_text)}"\n(ATUI="{atui}" SAB="{sab}")',
                meta={"def_vid": def_vid, "ATUI": atui, "SAB": sab},
                trace=trace_obj,
            ))

        # 2) Dense defs
        for idx, s01 in dense_ranked[: self.dense_defs_keep]:
            d = dense_defs[idx]
            atui = d.get("ATUI", "")
            sab = d.get("SAB", "")
            def_text = d.get("DEF", "")

            eid = f"E{eid_counter}"; eid_counter += 1
            nodes.append(EvidenceNode(
                eid=eid,
                route="dense_def_faiss",
                score01=float(s01),
                text=f"Definition: {normalize_text(def_text)}\n(ATUI={atui} SAB={sab})",
                meta={"ATUI": atui, "SAB": sab},
                trace={"route": "dense_def_faiss", "ATUI": atui, "SAB": sab},
            ))

        # 3) Semantic-name evidence (wider pool, then pruned)
        #    We score these with BM25(joint_query) so random lab tests get dropped.
        sem_items = retrieval["top_dense"]
        sem_texts = [f"Semantic: {normalize_text(it['name'])}\n(SUI={it['sui']})" for it in sem_items]
        joint = build_joint_query(question, options) if options else build_question_only_query(question)
        sem_bm = bm25_scores(joint, sem_texts)
        sem_bm01 = minmax01(sem_bm)

        # create semantic nodes with bm25 score as initial score01
        sem_nodes: List[EvidenceNode] = []
        for it, s01 in sorted(zip(sem_items, sem_bm01.tolist()), key=lambda x: x[1], reverse=True)[: self.semantic_name_keep * 3]:
            eid = f"E{eid_counter}"; eid_counter += 1
            sem_nodes.append(EvidenceNode(
                eid=eid,
                route="kg_semantic_name_only",
                score01=float(s01),
                text=f"Semantic: {normalize_text(it['name'])}\n(SUI={it['sui']})",
                meta={"sui": it["sui"], "name": it["name"]},
                trace={"route": "kg_semantic_name_only", "seed_sui": it["sui"]},
            ))

        nodes.extend(sem_nodes)

        # 4) Global prune (mode-specific)
        nodes = prune_nodes(
            mode=mode,
            question=question,
            options=options,
            nodes=nodes,
            keep_k=self.prune_keep_k,
            min_keep=self.prune_min_keep,
        )

        # Re-number EIDs to keep them contiguous after prune (keeps prompts small/clean)
        renum = []
        for i, n in enumerate(nodes, start=1):
            renum.append(EvidenceNode(
                eid=f"E{i}",
                route=n.route,
                score01=n.score01,
                text=n.text,
                meta=n.meta,
                trace=n.trace
            ))
        return renum

    # ---- LLM ----
    def _llm_call_zip(self, system: str, user: str) -> Dict[str, Any]:
        raw = self.llm.chat(
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=self.temperature,
        )
        return extract_first_json_obj(raw) or {}

    def answer_mcq_zip_style(self, question: str, options: Dict[str, str], nodes: List[EvidenceNode]) -> MCQZipAnswer:
        option_keys = sorted(options.keys(), key=lambda x: int(x) if x.isdigit() else x)
        allowed_eids = [n.eid for n in nodes]

        kg_nodes = [n for n in nodes if n.route == "kg_sui_concept_def"]
        kg_def_eids = [n.eid for n in kg_nodes]
        kg_max = max([n.score01 for n in kg_nodes], default=0.0)

        # enforce KG only if strong KG def exists
        enforce_kg = bool(kg_def_eids) and (kg_max >= self.kg_min_sigmoid_to_enforce) and self.require_kg_def_if_available

        evidence_block = "\n\n".join([f"[{n.eid}] {n.text}" for n in nodes])

        system_base = (
            "Return ONLY valid JSON with keys: cop_index, answer, why_correct, why_others_incorrect, evidence_used.\n"
            f"cop_index must be one of {option_keys} or \"-1\".\n"
            "If cop_index == \"-1\": answer must be \"\".\n"
            "If cop_index != \"-1\": answer must exactly equal the chosen option text.\n"
            "Cite evidence like [E1].\n"
            "IMPORTANT:\n"
            "- evidence_used MUST equal the set of [E#] citations you used.\n"
            "- why_others_incorrect MUST contain a non-empty reason for every option key other than the chosen one.\n"
        )

        if enforce_kg:
            system_policy = (
                "High-confidence KG definitions are available.\n"
                "You MUST cite at least one kg_sui_concept_def evidence ID.\n"
                f"Valid kg_sui_concept_def IDs: {kg_def_eids}.\n"
                "If KG definitions do not support a single option, output cop_index=\"-1\".\n"
            )
            fallback_reason = "Insufficient KG definition evidence to select a single option."
            fallback_ev = (kg_def_eids[:1] if kg_def_eids else [nodes[0].eid] if nodes else None)
        else:
            system_policy = (
                "KG definitions are absent or not confident.\n"
                "Still answer using the evidence list.\n"
                "You MAY use general medical knowledge, but MUST cite at least one evidence item.\n"
                "Only output cop_index=\"-1\" if you truly cannot choose.\n"
            )
            fallback_reason = "Insufficient evidence to select a single option."
            fallback_ev = ([nodes[0].eid] if nodes else None)

        user = f"""
Question:
{question}

Options (keys -> text):
{json.dumps(options, ensure_ascii=False)}

Evidence:
{evidence_block}

Return ONLY JSON:
- cop_index must be one of {option_keys} or "-1"
- if cop_index == "-1": answer must be ""
- if cop_index != "-1": answer must exactly equal the chosen option text
- cite evidence like [E1]
- evidence_used MUST list cited evidence IDs (e.g., ["E6","E9"])
- why_others_incorrect: MUST explain why EACH other option is incorrect (no empty strings)
""".strip()

        obj = self._llm_call_zip(system=f"{system_base}\n{system_policy}", user=user)

        ans = sanitize_zip_answer(
            obj=obj,
            options=options,
            allowed_eids=allowed_eids,
            fallback_reason=fallback_reason,
            fallback_evidence=fallback_ev,
        )

        # backfill evidence_used from citations if needed
        if not ans.evidence_used:
            cited = extract_eids_from_text(ans.why_correct, *list(ans.why_others_incorrect.values()))
            cited = [e for e in cited if e in allowed_eids]
            if cited:
                ans.evidence_used = cited

        if not ans.evidence_used and nodes:
            ans.evidence_used = [nodes[0].eid]
            if f"[{nodes[0].eid}]" not in (ans.why_correct or ""):
                ans.why_correct = (ans.why_correct.strip() + f" [{nodes[0].eid}]").strip()

        # enforce KG requirement if active
        if enforce_kg and not set(ans.evidence_used).intersection(set(kg_def_eids)):
            if self.retry_strict_once:
                strict = (
                    f"{system_base}\n"
                    "FINAL WARNING: Cite at least one kg_sui_concept_def evidence ID, OR abstain.\n"
                    f"Valid kg_sui_concept_def IDs: {kg_def_eids}.\n"
                )
                obj2 = self._llm_call_zip(system=strict, user=user)
                ans2 = sanitize_zip_answer(
                    obj=obj2,
                    options=options,
                    allowed_eids=allowed_eids,
                    fallback_reason="Insufficient KG definition evidence to select a single option.",
                    fallback_evidence=(kg_def_eids[:1] if kg_def_eids else fallback_ev),
                )
                if ans2.cop_index == "-1" or set(ans2.evidence_used).intersection(set(kg_def_eids)):
                    ans2.evidence = build_evidence_payload(nodes, ans2.evidence_used)
                    return ans2

            forced = MCQZipAnswer(
                cop_index="-1",
                answer="",
                why_correct=f"Insufficient KG definition evidence to select a single option. [{kg_def_eids[0]}]",
                why_others_incorrect={k: "" for k in options.keys()},
                evidence_used=[kg_def_eids[0]],
                evidence={}
            )
            forced.evidence = build_evidence_payload(nodes, forced.evidence_used)
            return forced

        ans.evidence = build_evidence_payload(nodes, ans.evidence_used)
        return ans

    def run_one(
        self,
        question: str,
        options: Dict[str, str],
        *,
        mode: str = "mcq",
        print_debug: bool = True
    ) -> Dict[str, Any]:
        mode = (mode or "mcq").lower().strip()
        joint_query = build_joint_query(question, options) if options else build_question_only_query(question)

        retrieval = self.retrieve_seeds(joint_query)

        dense_defs = self.dense_def_candidates(joint_query)
        dense_ranked = self.rerank_dense_defs(joint_query, dense_defs)

        kg = self.kg_defs_from_seeds(retrieval["seeds_mmr"])
        kg_ranked = self.rerank_kg_defs(joint_query, kg["def_props"])

        nodes = self.build_evidence_nodes(
            retrieval=retrieval,
            kg=kg,
            kg_ranked=kg_ranked,
            dense_defs=dense_defs,
            dense_ranked=dense_ranked,
            question=question,
            options=options,
            mode=mode,
        )

        ans = self.answer_mcq_zip_style(question, options, nodes)

        if print_debug:
            print("Evidence nodes:", len(nodes))
            print("\n--- Route counts in nodes ---")
            print(Counter([n.route for n in nodes]))

            print("\n--- LAST DEF QUERY (only) ---")
            print(kg.get("last_fetch_def_query", "") or "(none)")

            print("\n--- Top evidence ---")
            for n in nodes[: min(12, len(nodes))]:
                print(f"[{n.eid}] score={n.score01:.4f} route={n.route}\n{n.text}\n")

            print("\n--- Prediction ---")
            print(ans.model_dump())

        return {
            "answer": ans.model_dump(),
            "evidence_used_traces": {
                eid: next((n.trace for n in nodes if n.eid == eid), None)
                for eid in ans.evidence_used
            },
        }

# ----------------------------
# Build pipeline
# ----------------------------
def build_pipeline() -> KGTraceableMCQPipeline:
    sem_index_path = str(Path(DATA_DIR) / SEM_INDEX_FILE)
    sem_meta_path  = str(Path(DATA_DIR) / SEM_META_JSONL)
    def_index_path = str(Path(DATA_DIR) / DEF_INDEX_FILE)
    def_meta_path  = str(Path(DATA_DIR) / DEF_META_FILE)

    for p in [sem_index_path, sem_meta_path, def_index_path, def_meta_path]:
        if not Path(p).exists():
            raise RuntimeError(f"Missing required file: {p}")

    # CUDA ALWAYS (no env var checks, no silent CPU)
    embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
    cross = CrossEncoder(CROSS_ENCODER_MODEL, device="cuda")

    sem_faiss = FaissIndex(sem_index_path, sem_meta_path)
    def_faiss = FaissIndex(def_index_path, def_meta_path)

    nebula = NebulaClient(
        host=NEBULA_HOST,
        port=NEBULA_PORT,
        user=NEBULA_USER,
        password=NEBULA_PASSWORD,
        space=NEBULA_SPACE,
    )

    llm = OpenAILikeOllama(
        base_url=OPENAI_BASE_URL,
        model=MODEL_NAME,
        api_key=API_KEY,
        timeout_s=120,
    )

    return KGTraceableMCQPipeline(
        semantic_faiss=sem_faiss,
        def_faiss=def_faiss,
        embedder=embedder,
        cross_encoder=cross,
        nebula=nebula,
        llm=llm,
        semantic_topk_dense=220,
        seed_k_mmr=60,
        mmr_lambda=0.7,
        dense_def_topk=80,
        dense_defs_keep=20,
        concept_limit_each_sem=200,
        def_pairs_limit_total=6000,
        kg_defs_keep=25,
        semantic_name_keep=10,
        temperature=0.0,
        require_kg_def_if_available=True,
        kg_min_sigmoid_to_enforce=0.70,
        retry_strict_once=True,
        prune_keep_k=32,
        prune_min_keep=4,
    )

# ----------------------------
# Runner
# ----------------------------
def run_mcq(
    row_data: Dict[str, Any],
    *,
    pipe: Optional[KGTraceableMCQPipeline] = None,
    mode: str = "mcq",
    print_debug: bool = True
) -> Dict[str, Any]:
    question = str(row_data.get("question", "")).strip()
    options = safe_literal_eval_dict(row_data.get("options", {}))
    if pipe is None:
        pipe = build_pipeline()
    return pipe.run_one(question, options, mode=mode, print_debug=print_debug)

# ----------------------------
# Example
# ----------------------------
if __name__ == "__main__":
    pipe = build_pipeline()

    row_data_1 = {
        "question": "Bonney's test is used determine ?",
        "options": "{'0': 'Uterine prolapsed', '1': 'Stress urinary incontinence', '2': 'Vesicovaginal fistula', '3': 'Ureteric fistula'}"
    }
    _ = run_mcq(row_data_1, pipe=pipe, mode="mcq", print_debug=True)

    row_data_2 = {
        "question": "Peripheral chemoreceptors are not sensitive to which of the following?",
        "options": "{'0': 'Hypoxia', '1': 'Hypercapnoea', '2': 'Hypocapnoea', '3': 'Dissolved H+ ions in blood'}"
    }
    _ = run_mcq(row_data_2, pipe=pipe, mode="mcq", print_debug=True)

    # Chatbot-safe (option-agnostic) mode would look like:
    # _ = pipe.run_one("Bonney's test is used determine ?", {}, mode="chat", print_debug=True)

    try:
        pipe.nebula.close()
    except Exception:
        pass


/home/macharya/dev/medkg-eval/graphrag_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
INFO:nebula3.logger:Get connection to ('127.0.0.1', 9669)
Batches: 100%|██████████| 1/1 [00:00<00:00, 26.87it/s]
INFO:httpx:HTTP Request: POST https://ollama.zib.de/api/chat/completions "HTTP/1.1 200 OK"


Evidence nodes: 32

--- Route counts in nodes ---
Counter({'dense_def_faiss': 20, 'kg_semantic_name_only': 9, 'kg_sui_concept_def': 3})

--- LAST DEF QUERY (only) ---
FETCH PROP ON Definition "AT198001346", "AT20135097", "AT271242363"
YIELD id(vertex) AS def_vid, Definition.ATUI AS atui, Definition.DEF AS def_text, Definition.SAB AS sab;

--- Top evidence ---
[E1] score=0.9843 route=kg_semantic_name_only
Semantic: Bonney's stress incontin test
(SUI=S0821343)

[E2] score=0.7801 route=dense_def_faiss
Definition: An abnormal anatomical passage that connects the VAGINA to other organs, such as the bladder (VESICOVAGINAL FISTULA) or the rectum (RECTOVAGINAL FISTULA).
(ATUI=AT38136791 SAB=MSH)

[E3] score=0.7570 route=dense_def_faiss
Definition: Placement and monitoring of a vaginal device for treating stress urinary incontinence, uterine retroversion, genital prolapse, or incompetent cervix
(ATUI=AT236299950 SAB=NIC)

[E4] score=0.5126 route=dense_def_faiss
Definition: The presence of a fis

Batches: 100%|██████████| 1/1 [00:00<00:00, 64.54it/s]
INFO:httpx:HTTP Request: POST https://ollama.zib.de/api/chat/completions "HTTP/1.1 200 OK"


Evidence nodes: 32

--- Route counts in nodes ---
Counter({'dense_def_faiss': 20, 'kg_semantic_name_only': 9, 'kg_sui_concept_def': 3})

--- LAST DEF QUERY (only) ---
FETCH PROP ON Definition "AT137259211", "AT148903557", "AT271204682"
YIELD id(vertex) AS def_vid, Definition.ATUI AS atui, Definition.DEF AS def_text, Definition.SAB AS sab;

--- Top evidence ---
[E1] score=0.7681 route=kg_semantic_name_only
Semantic: Carbon dioxide 0.2 mL/mL and oxygen 0.8 mL/mL gas for inhalation
(SUI=S20322345)

[E2] score=0.6476 route=dense_def_faiss
Definition: A state of general hypoxia and hypercapnea, resulting in acidosis, which affects all tissues in the body.
(ATUI=AT205721560 SAB=NCI)

[E3] score=0.6413 route=kg_sui_concept_def
Definition: "The process in which information about the levels of carbon dioxide are received and are converted to a molecular signal by chemoreceptors in a carotid body. [GOC:mtg_cardio]"
(ATUI="AT137259211" SAB="GO")

[E4] score=0.6372 route=dense_def_faiss
Definition

In [ ]:
"Q: In the ion chromatography technique with suppressor column, the suppressor column is located in the chromatograph:	"
{'0': 'Before the ion exchange separation column.', 
'1': 'After the ion exchange separation column.', 
'2': 'After the detector.', 
'3': 'Before the injection valve.'}


In [ ]:
{
    "id": "932f2bf9-05a6-4ad2-a35c-05b12c2dd16a",
    "testbed_data": {
      "correct_index": "1"
    },
    "gpt_output": {
      "cop_index": "1",
      "answer": "After the ion exchange separation column.",
      "why_correct": "[E1],[E2]  describe Ion Chromatography using a stationary phase of ion exchange resins. [E15], [E19] describe how the process works, with the sample being introduced, and then separated based on interaction with the stationary phase. This implies the suppressor column must come after the initial separation.",
      "why_others_incorrect": {
        "0": "The suppressor column doesn't precede the ion exchange separation column; it acts to remove unwanted ions after separation.",
        "2": "The suppressor column is specifically designed to handle ionic components separated by the chromatography process, not for tasks after detection.",
        "3": "The sample injection occurs before the ion exchange separation column. The suppressor column functions later in the process."
      }
    }
  }

In [ ]:
{
    "id": "932f2bf9-05a6-4ad2-a35c-05b12c2dd16a",
    "evidence_used": [
      "E1",
      "E2",
      "E15",
      "E19"
    ],
    "all_evidence": [
      {
        "eid": "E1",
        "score": 0.7377802133560181,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"Separation technique in which the stationary phase consists of ion exchange resins. The resins contain loosely held small ions that easily exchange places with other small ions of like charge present in solutions washed over the resins.\"\n(ATUI=\"AT38136926\" SAB=\"MSH\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S0025182",
          "sem_vid": "S0025182",
          "concept_vid": "C0008564",
          "def_vid": "AT38136926",
          "atui": "AT38136926",
          "sab": "MSH"
        },
        "ATUI": "AT38136926",
        "SAB": "MSH"
      },
      {
        "eid": "E2",
        "score": 0.7377588748931885,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"separation technique in which the stationary phase consists of ion exchange resins; the resins contain loosely held small ions that easily exchange places with other small ions of like charge present in solutions washed over the resins.\"\n(ATUI=\"AT51224340\" SAB=\"CSP\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S0025182",
          "sem_vid": "S0025182",
          "concept_vid": "C0008564",
          "def_vid": "AT51224340",
          "atui": "AT51224340",
          "sab": "CSP"
        },
        "ATUI": "AT51224340",
        "SAB": "CSP"
      },
      {
        "eid": "E3",
        "score": 0.734555721282959,
        "route": "dense_def_faiss",
        "text": "Definition: Separation technique in which the stationary phase consists of ion exchange resins. The resins contain loosely held small ions that easily exchange places with other small ions of like charge present in solutions washed over the resins.\n(ATUI=AT38136926 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT38136926",
          "SAB": "MSH"
        },
        "ATUI": "AT38136926",
        "SAB": "MSH"
      },
      {
        "eid": "E4",
        "score": 0.7345343828201294,
        "route": "dense_def_faiss",
        "text": "Definition: separation technique in which the stationary phase consists of ion exchange resins; the resins contain loosely held small ions that easily exchange places with other small ions of like charge present in solutions washed over the resins.\n(ATUI=AT51224340 SAB=CSP)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT51224340",
          "SAB": "CSP"
        },
        "ATUI": "AT51224340",
        "SAB": "CSP"
      },
      {
        "eid": "E5",
        "score": 0.7191727161407471,
        "route": "dense_def_faiss",
        "text": "Definition: The analysis of a chemical substance by inserting a sample into a carrier stream of reagent using a sample injection valve that propels the sample downstream where mixing occurs in a coiled tube, then passes into a flow-through detector and a recorder or other data handling device.\n(ATUI=AT38144084 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT38144084",
          "SAB": "MSH"
        },
        "ATUI": "AT38144084",
        "SAB": "MSH"
      },
      {
        "eid": "E6",
        "score": 0.7057665586471558,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Chromatography Systems, Liquid, Packed Column, High-Pressure, Ion-Exchange\n(SUI=S1451070)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S1451070"
        }
      },
      {
        "eid": "E7",
        "score": 0.6733829975128174,
        "route": "dense_def_faiss",
        "text": "Definition: Chromatography systems in which the separation process is accomplished by forcing a gaseous mixture of analytes as the mobile phase (e.g., an inert gas, such as nitrogen or helium) through a column of the stationary phase. Separation of solutes in the mixture is based on the relative differences in their vapor pressures and their interaction with the stationary phase. A basic gas chromatograph consists of a gas reservoir and flow control device, an injector, the separation column, an oven, an online detector (sometimes a mass spectrometer), and, usually, a computer.\n(ATUI=AT271197790 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271197790",
          "SAB": "UMD"
        },
        "ATUI": "AT271197790",
        "SAB": "UMD"
      },
      {
        "eid": "E8",
        "score": 0.643779456615448,
        "route": "dense_def_faiss",
        "text": "Definition: Laboratory syringes designed to inject samples in the injection ports of gas and/or liquid chromatography systems. These devices typically consist of small-volume syringes that include either fixed needles (e.g., cemented), Luer-lock tips, or other special tips. Gas/liquid chromatography syringes are available to inject the samples in a specific injection port, such as the split/splitless or the packed-column injection port; some syringes are manufactured for use only in devices of a particular model or from a specific manufacturer .\n(ATUI=AT271200229 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271200229",
          "SAB": "UMD"
        },
        "ATUI": "AT271200229",
        "SAB": "UMD"
      },
      {
        "eid": "E9",
        "score": 0.6388394832611084,
        "route": "dense_def_faiss",
        "text": "Definition: A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\n(ATUI=AT20133085 SAB=SPN)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT20133085",
          "SAB": "SPN"
        },
        "ATUI": "AT20133085",
        "SAB": "SPN"
      },
      {
        "eid": "E10",
        "score": 0.6388394832611084,
        "route": "dense_def_faiss",
        "text": "Definition: A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\n(ATUI=AT20133084 SAB=SPN)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT20133084",
          "SAB": "SPN"
        },
        "ATUI": "AT20133084",
        "SAB": "SPN"
      },
      {
        "eid": "E11",
        "score": 0.6388394832611084,
        "route": "dense_def_faiss",
        "text": "Definition: A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\n(ATUI=AT20133083 SAB=SPN)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT20133083",
          "SAB": "SPN"
        },
        "ATUI": "AT20133083",
        "SAB": "SPN"
      },
      {
        "eid": "E12",
        "score": 0.6388394832611084,
        "route": "dense_def_faiss",
        "text": "Definition: A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\n(ATUI=AT20133082 SAB=SPN)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT20133082",
          "SAB": "SPN"
        },
        "ATUI": "AT20133082",
        "SAB": "SPN"
      },
      {
        "eid": "E13",
        "score": 0.6385672092437744,
        "route": "dense_def_faiss",
        "text": "Definition: Packed-column liquid chromatography systems that use a relatively high pressure to pump the liquid through the column and in which the stationary phase works with high efficiency (i.e., a high-performance packed column). These systems are usually classified according to the principle used in separating components; the most used techniques are adsorption chromatography, bonded-phase (or partition) chromatography (there are two types of bonded-phase technologies--normal phase and the more commonly used reverse phase), ion-exchange chromatography, and steric-exclusion (or gel permeation) chromatography.\n(ATUI=AT271205039 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271205039",
          "SAB": "UMD"
        },
        "ATUI": "AT271205039",
        "SAB": "UMD"
      },
      {
        "eid": "E14",
        "score": 0.6336822509765625,
        "route": "dense_def_faiss",
        "text": "Definition: A process used for separating mixtures by virtue of differences in absorbency. It involves stationary and mobile phases. The stationary phase was packed in a column with materials that can be of any partitioning capability, adsorption, ion exchange, or affinity. The mobile phase (either liquid or gas) is the mixture that required to be separated.\n(ATUI=AT198126806 SAB=NCI)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT198126806",
          "SAB": "NCI"
        },
        "ATUI": "AT198126806",
        "SAB": "NCI"
      },
      {
        "eid": "E15",
        "score": 0.6331939697265625,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"Chromatography systems in which the separation process of the mixture of compounds is accomplished by pumping a liquid, usually water or an organic solvent (i.e., the mobile phase) through a column of stationary phase. Separation is based on the relative solubility of the solutes in the two phases and their interaction with the stationary phase. A mass spectrometer can be connected to these devices instead of a detector.\"\n(ATUI=\"AT271226909\" SAB=\"UMD\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S1451092",
          "sem_vid": "S1451092",
          "concept_vid": "C0179912",
          "def_vid": "AT271226909",
          "atui": "AT271226909",
          "sab": "UMD"
        },
        "ATUI": "AT271226909",
        "SAB": "UMD"
      },
      {
        "eid": "E16",
        "score": 0.62806236743927,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\"\n(ATUI=\"AT20133082\" SAB=\"SPN\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S1451078",
          "sem_vid": "S1451078",
          "concept_vid": "C0687342",
          "def_vid": "AT20133082",
          "atui": "AT20133082",
          "sab": "SPN"
        },
        "ATUI": "AT20133082",
        "SAB": "SPN"
      },
      {
        "eid": "E17",
        "score": 0.62806236743927,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"A chromatographic separation material for clinical use is a device accessory (e.g., ion exchange absorbents, ion exchagne resins, and ion papers) intended for use in ion exchange chromatography, a procedure in which a compound is separated from a solution.\"\n(ATUI=\"AT20133083\" SAB=\"SPN\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S0788536",
          "sem_vid": "S0788536",
          "concept_vid": "C0490507",
          "def_vid": "AT20133083",
          "atui": "AT20133083",
          "sab": "SPN"
        },
        "ATUI": "AT20133083",
        "SAB": "SPN"
      },
      {
        "eid": "E18",
        "score": 0.6279996633529663,
        "route": "dense_def_faiss",
        "text": "Definition: Chromatography systems in which the separation process of the mixture of compounds is accomplished by pumping a liquid, usually water or an organic solvent (i.e., the mobile phase) through a column of stationary phase. Separation is based on the relative solubility of the solutes in the two phases and their interaction with the stationary phase. A mass spectrometer can be connected to these devices instead of a detector.\n(ATUI=AT271226909 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271226909",
          "SAB": "UMD"
        },
        "ATUI": "AT271226909",
        "SAB": "UMD"
      },
      {
        "eid": "E19",
        "score": 0.6274697780609131,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"Packed-column liquid chromatography systems that use a relatively high pressure to pump the liquid through the column and in which the stationary phase works with high efficiency (i.e., a high-performance packed column). These systems are usually classified according to the principle used in separating components; the most used techniques are adsorption chromatography, bonded-phase (or partition) chromatography (there are two types of bonded-phase technologies--normal phase and the more commonly used reverse phase), ion-exchange chromatography, and steric-exclusion (or gel permeation) chromatography.\"\n(ATUI=\"AT271205039\" SAB=\"UMD\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S1451074",
          "sem_vid": "S1451074",
          "concept_vid": "C0687276",
          "def_vid": "AT271205039",
          "atui": "AT271205039",
          "sab": "UMD"
        },
        "ATUI": "AT271205039",
        "SAB": "UMD"
      },
      {
        "eid": "E20",
        "score": 0.6252028346061707,
        "route": "dense_def_faiss",
        "text": "Definition: Chromatography injectors that are operated manually and are sample-delivery systems designed to interface with gas-liquid chromatography or high-pressure liquid chromatography systems to introduce gas, liquid, or solid phase micro extraction samples into the sample loop (mobile phase) of the chromatography system in the smallest volume possible so that the sample enters the stationary phase column as a homogeneous, low-volume plug. They come in different shapes and sizes to fit into various chromatography units and typically feature an injection port to receive the sample microsyringe while in injection mode, and outflow ports that interface with the stator injection port of the chromatography unit to inject the sample into the chromatography unit while in outflow mode. The injector is manually switched between the two modes by the operator by rotating the handle of the injector. The manual injectors usually have tight seals between interface ports and are made of stainless steel or other noncorrosive materials to avoid contamination of samples and wear of the injector components by the sample material. Manual injectors are usually used when only one or a few samples need to be introduced into the chromatography unit in a fast and uniform manner at a controlled pressure.\n(ATUI=AT271250249 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271250249",
          "SAB": "UMD"
        },
        "ATUI": "AT271250249",
        "SAB": "UMD"
      },
      {
        "eid": "E21",
        "score": 0.6248865127563477,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"A process used for separating mixtures by virtue of differences in absorbency. It involves stationary and mobile phases. The stationary phase was packed in a column with materials that can be of any partitioning capability, adsorption, ion exchange, or affinity. The mobile phase (either liquid or gas) is the mixture that required to be separated.\"\n(ATUI=\"AT198126806\" SAB=\"NCI\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S6147895",
          "sem_vid": "S6147895",
          "concept_vid": "C0684296",
          "def_vid": "AT198126806",
          "atui": "AT198126806",
          "sab": "NCI"
        },
        "ATUI": "AT198126806",
        "SAB": "NCI"
      },
      {
        "eid": "E22",
        "score": 0.6166471242904663,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Chromatography, Ion Exchange\n(SUI=S0025182)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0025182"
        }
      },
      {
        "eid": "E23",
        "score": 0.6166471242904663,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Chromatography, Ion-Exchange\n(SUI=S0025183)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0025183"
        }
      },
      {
        "eid": "E24",
        "score": 0.6166471242904663,
        "route": "kg_semantic_name_only",
        "text": "Semantic: ION-EXCHANGE CHROMATOGRAPHY\n(SUI=S0908506)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0908506"
        }
      },
      {
        "eid": "E25",
        "score": 0.6166471242904663,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Ion-Exchange Chromatography\n(SUI=S0054290)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0054290"
        }
      },
      {
        "eid": "E26",
        "score": 0.6142213940620422,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"Detectors designed to assess the individual molecules that come out (i.e., elute) from a chromatography column usually by detecting changes in the properties of the mobile phase and/or the sample constituents. There is a great variety of chromatography detectors either designed and manufactured to detect eluates for liquid and high-performance liquid chromatograph (HPLC) columns or from gas chromatograph columns. The optimum type of detector frequently depends on the characteristics of the eluate (e.g., electrolytes, proteins, lipids). Chromatography detectors are mainly used in clinical laboratories as a component of chromatography systems intended to assess components in body fluids or other compounds; the detectors are also used in healthcare research.\"\n(ATUI=\"AT271234455\" SAB=\"UMD\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S14595825",
          "sem_vid": "S14595825",
          "concept_vid": "C3856868",
          "def_vid": "AT271234455",
          "atui": "AT271234455",
          "sab": "UMD"
        },
        "ATUI": "AT271234455",
        "SAB": "UMD"
      },
      {
        "eid": "E27",
        "score": 0.6119365692138672,
        "route": "dense_def_faiss",
        "text": "Definition: Sample delivery systems designed to interface with gas-liquid chromatography systems or high-pressure liquid chromatography systems to introduce gas, liquid or solid phase micro extraction samples into the sample loop (mobile phase) of the system in the smallest volume possible so that the sample enters the stationary phase column as a homogeneous, low-volume plug. Chromatography injectors can be manual or automated. Some chromatography injectors have a microsyringe that is lowered into a sample vial through a septum to draw gas, liquid or solid phase micro extraction samples into the syringe in measured amounts. After the syringe injector is filled, the chromatography injectors are interfaced with the injection port of the chromatography unit to introduce the sample. Both automated injectors and manual injectors are available for use with gas chromatography and high-pressure liquid chromatography systems.\n(ATUI=AT271235675 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271235675",
          "SAB": "UMD"
        },
        "ATUI": "AT271235675",
        "SAB": "UMD"
      },
      {
        "eid": "E28",
        "score": 0.6072167754173279,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Ion-Exchange Chromatographies\n(SUI=S0054289)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0054289"
        }
      },
      {
        "eid": "E29",
        "score": 0.6072167754173279,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Chromatographies, Ion-Exchange\n(SUI=S0025159)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0025159"
        }
      },
      {
        "eid": "E30",
        "score": 0.6072167754173279,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Chromatographies, Ion Exchange\n(SUI=S0025158)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0025158"
        }
      },
      {
        "eid": "E31",
        "score": 0.6072167754173279,
        "route": "kg_semantic_name_only",
        "text": "Semantic: ADSORBENTS, ION-EXCHANGE\n(SUI=S0788536)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0788536"
        }
      },
      {
        "eid": "E32",
        "score": 0.6064043045043945,
        "route": "dense_def_faiss",
        "text": "Definition: Detectors designed to assess the individual molecules that come out (i.e., elute) from a chromatography column usually by detecting changes in the properties of the mobile phase and/or the sample constituents. There is a great variety of chromatography detectors either designed and manufactured to detect eluates for liquid and high-performance liquid chromatograph (HPLC) columns or from gas chromatograph columns. The optimum type of detector frequently depends on the characteristics of the eluate (e.g., electrolytes, proteins, lipids). Chromatography detectors are mainly used in clinical laboratories as a component of chromatography systems intended to assess components in body fluids or other compounds; the detectors are also used in healthcare research.\n(ATUI=AT271234455 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271234455",
          "SAB": "UMD"
        },
        "ATUI": "AT271234455",
        "SAB": "UMD"
      }
    ]
  }

In [ ]:
"Q: Which of the following bacteria can be a cause of rheumatic fever?	"
{'0': 'Nocardia brasiliensis', 
 '1': 'Streptococcus pyogenes.', 
 '2': 'Staphylococcus epidermidis.', 
 '3': 'Enterococcus faecium.'}


In [ ]:
{
    "id": "b429bdb8-9dfb-4265-bdd2-c5465a604732",
    "testbed_data": {
      "correct_index": "1"
    },
    "gpt_output": {
      "cop_index": "1",
      "answer": "Streptococcus pyogenes.",
      "why_correct": "[E2, E7, E13, E18, E24, E25, E30] all state that Streptococcus pyogenes is a bacterium that can cause rheumatic fever. [E1]",
      "why_others_incorrect": {
        "0": "Nocardia brasiliensis is associated with infections like Nocardiosis but not Rheumatic Fever. [E3, E4, E8]",
        "2": "Staphylococcus epidermidis is commonly found on the skin and can cause infections but not typically Rheumatic Fever.",
        "3": "Enterococcus faecium is a bacterium that can cause infections but is not known to be associated with Rheumatic Fever."
      }
    }
  }

In [ ]:
{
    "id": "b429bdb8-9dfb-4265-bdd2-c5465a604732",
    "evidence_used": [
      "E1"
    ],
    "all_evidence": [
      {
        "eid": "E1",
        "score": 0.7958292365074158,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents used in rapid tests to detect Streptococcus class A (Streptococcus pyogenes), a species of coccoid bacteria, in a patient's sample. Streptococcus pyogenes is one of the most important bacterial pathogens of humans, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271228220 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271228220",
          "SAB": "UMD"
        },
        "ATUI": "AT271228220",
        "SAB": "UMD"
      },
      {
        "eid": "E2",
        "score": 0.772033154964447,
        "route": "dense_def_faiss",
        "text": "Definition: A species of gram-positive, coccoid bacteria isolated from skin lesions, blood, inflammatory exudates, and the upper respiratory tract of humans. It is a group A hemolytic Streptococcus that can cause SCARLET FEVER and RHEUMATIC FEVER.\n(ATUI=AT53894777 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT53894777",
          "SAB": "MSH"
        },
        "ATUI": "AT53894777",
        "SAB": "MSH"
      },
      {
        "eid": "E3",
        "score": 0.7598804235458374,
        "route": "dense_def_faiss",
        "text": "Definition: Infections with bacteria of the genus NOCARDIA.\n(ATUI=AT38154428 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT38154428",
          "SAB": "MSH"
        },
        "ATUI": "AT38154428",
        "SAB": "MSH"
      },
      {
        "eid": "E4",
        "score": 0.7383304834365845,
        "route": "dense_def_faiss",
        "text": "Definition: gram positive bacterial infection with bacteria of the genus Nocardia.\n(ATUI=AT32350619 SAB=CSP)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT32350619",
          "SAB": "CSP"
        },
        "ATUI": "AT32350619",
        "SAB": "CSP"
      },
      {
        "eid": "E5",
        "score": 0.7350855469703674,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents intended to detect antibodies to the M protein produced by Streptococcus group A (typically Streptococcus pyogenes), a spherical bacteria of the family Streptococcaceae. Streptococcus pyogenes is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever). These antibodies identify specific groups of S. and may be associated with some autoimmune diseases (such as acute rheumatic fever) idiopathic mitral regurgitation, and poststreptococcal glomerulonephritis.\n(ATUI=AT271199028 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271199028",
          "SAB": "UMD"
        },
        "ATUI": "AT271199028",
        "SAB": "UMD"
      },
      {
        "eid": "E6",
        "score": 0.7207925319671631,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents used in tests intended to detect antigens that permit the identification of strains of Streptococcus, a genus of spherical bacteria of the family Streptococcaceae. Streptococci that are pathologic in humans include S. pneumoniae (pneumococcus), a major respiratory tract pathogen and the most common cause of pneumonia, and S. pyogenes, which is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271213698 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271213698",
          "SAB": "UMD"
        },
        "ATUI": "AT271213698",
        "SAB": "UMD"
      },
      {
        "eid": "E7",
        "score": 0.7080563902854919,
        "route": "dense_def_faiss",
        "text": "Definition: species of gram-positive, coccoid bacteria isolated from skin lesions, blood, inflammatory exudates, and the upper respiratory tract of humans; it is a group A hemolytic Streptococcus that can cause scarlet fever and rheumatic fever; virulent strains penetrate deep into the body, with catastrophic results; it has been demonstrated that invasive streptococcus A infections can trigger a toxic shock syndrome, myositis, or destroy the sheath that covers the muscle, necrotizing fasciitis.\n(ATUI=AT51220972 SAB=CSP)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT51220972",
          "SAB": "CSP"
        },
        "ATUI": "AT51220972",
        "SAB": "CSP"
      },
      {
        "eid": "E8",
        "score": 0.698153018951416,
        "route": "dense_def_faiss",
        "text": "Definition: A bacterial infection caused by members of the gram-positive bacilli genus Nocardia.\n(ATUI=AT262361888 SAB=NCI)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT262361888",
          "SAB": "NCI"
        },
        "ATUI": "AT262361888",
        "SAB": "NCI"
      },
      {
        "eid": "E9",
        "score": 0.6975361108779907,
        "route": "dense_def_faiss",
        "text": "Definition: Molecular assay reagents intended to identify species of Streptococcus, a genus of spherical bacteria of the family Streptococcaceae, by detecting specific nucleic-acid information of the target bacteria. Streptococci tend to grow in pairs or chains and are divided into serogenic types according to their cell wall (e.g., groups A, B, C). Streptococci that are pathologic to humans include S. pneumoniae, a major respiratory tract pathogen and the most common cause of pneumonia, and S. pyogenes, which one of the most important bacterial human pathogens and causes a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271213610 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271213610",
          "SAB": "UMD"
        },
        "ATUI": "AT271213610",
        "SAB": "UMD"
      },
      {
        "eid": "E10",
        "score": 0.6681522727012634,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents intended to detect antibodies to exotoxins produced by several antigenically distinct types of Streptococcus group A (typically Streptococcus pyogenes), spherical bacteria of the family Streptococcaceae. S. pyogenes is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever). These antibodies are associated with some rheumatic autoimmune diseases.\n(ATUI=AT271220993 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271220993",
          "SAB": "UMD"
        },
        "ATUI": "AT271220993",
        "SAB": "UMD"
      },
      {
        "eid": "E11",
        "score": 0.656552791595459,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents intended to detect antibodies to species of Streptococcus, a genus of spherical bacteria of the family Streptococcaceae. Streptococci tend to grow in pairs or chains and are divided into many serogenic types according to their cell wall (e.g., groups A, B, and C). Streptococci that are pathogenic in humans include many species, such as S. pneumoniae (pneumococcus), a major respiratory tract pathogen and the most common cause of pneumonia, and S. pyogenes, which causes a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271199030 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271199030",
          "SAB": "UMD"
        },
        "ATUI": "AT271199030",
        "SAB": "UMD"
      },
      {
        "eid": "E12",
        "score": 0.654213547706604,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents intended to detect antibody titers or antigens of species of Streptococcus, a genus of spherical bacteria of the family Streptococcaceae. Streptococci tend to grow in pairs or chains and are divided into many serogenic types according to their cell wall (e.g., groups A, B, C). Streptococci that are pathologic to humans include S. pneumoniae (pneumococcus), a major respiratory tract pathogen and the most common cause of pneumonia, and S. pyogenes, which is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271250113 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271250113",
          "SAB": "UMD"
        },
        "ATUI": "AT271250113",
        "SAB": "UMD"
      },
      {
        "eid": "E13",
        "score": 0.6537244319915771,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"Rheumatic fever (RF) is a multisystem inflammatory disease occurring as a post-infectious, nonsuppurative sequela of untreated streptococcus pyogenes (Group A streptococcus [GAS]) pharyngitis, and mainly occurs in individuals aged 5 to 15 years. The most common presenting signs are fever, migratory polyarthritis and carditis.\"\n(ATUI=\"AT277511376\" SAB=\"ORPHANET\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S0082534",
          "sem_vid": "S0082534",
          "concept_vid": "C0035436",
          "def_vid": "AT277511376",
          "atui": "AT277511376",
          "sab": "ORPHANET"
        },
        "ATUI": "AT277511376",
        "SAB": "ORPHANET"
      },
      {
        "eid": "E14",
        "score": 0.6492102742195129,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents used in tests intended to detect antigens that permit the identification of strains of Streptococcus group A (typically Streptococcus pyogenes), spherical bacteria of the family Streptococcaceae. St. pyogenes is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever). These antigens are associated with some autoimmune diseases, including rheumatoid arthritis and osteoarthritis.\n(ATUI=AT271206385 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271206385",
          "SAB": "UMD"
        },
        "ATUI": "AT271206385",
        "SAB": "UMD"
      },
      {
        "eid": "E15",
        "score": 0.6491706967353821,
        "route": "dense_def_faiss",
        "text": "Definition: Microbiology reagents intended to identify group A Streptococcus (typically Streptococcus pyogenes), a species of spherical bacteria of the family Streptococcacaea. Group A streptococci produce a wide array of serious infections, including pharyngitis, impetigo, endocarditis, meningitis, and arthritis.\n(ATUI=AT271198630 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271198630",
          "SAB": "UMD"
        },
        "ATUI": "AT271198630",
        "SAB": "UMD"
      },
      {
        "eid": "E16",
        "score": 0.6309508085250854,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Fever episodes last from 3 to 10 days\n(SUI=S7506775)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S7506775"
        }
      },
      {
        "eid": "E17",
        "score": 0.6191564202308655,
        "route": "dense_def_faiss",
        "text": "Definition: Serology reagents intended to detect antibodies to deoxyribonuclease B (Dnase B antibodies) that develop in response to or as the result of a preceding Streptococcal infection; streptococcus is a genus of spherical bacteria of the family Streptococcaceae. Streptococci that are pathologic to humans include S. pneumoniae (pneumococcus), a major respiratory tract pathogen and the most common cause of pneumonia, and S. pyogenes, which is one of the most important bacterial human pathogens, causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever). Dnase B antibodies are associated with some autoimmune diseases, including rheumatic disorders (e.g., poststreptococcal reactive arthritis).\n(ATUI=AT271228192 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271228192",
          "SAB": "UMD"
        },
        "ATUI": "AT271228192",
        "SAB": "UMD"
      },
      {
        "eid": "E18",
        "score": 0.6190653443336487,
        "route": "dense_def_faiss",
        "text": "Definition: Rheumatic fever (RF) is a multisystem inflammatory disease occurring as a post-infectious, nonsuppurative sequela of untreated streptococcus pyogenes (Group A streptococcus [GAS]) pharyngitis, and mainly occurs in individuals aged 5 to 15 years. The most common presenting signs are fever, migratory polyarthritis and carditis.\n(ATUI=AT277511376 SAB=ORPHANET)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT277511376",
          "SAB": "ORPHANET"
        },
        "ATUI": "AT277511376",
        "SAB": "ORPHANET"
      },
      {
        "eid": "E19",
        "score": 0.601220428943634,
        "route": "kg_semantic_name_only",
        "text": "Semantic: history of rheumatic fever\n(SUI=S1458596)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S1458596"
        }
      },
      {
        "eid": "E20",
        "score": 0.601220428943634,
        "route": "kg_semantic_name_only",
        "text": "Semantic: History of rheumatic fever\n(SUI=S0224946)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0224946"
        }
      },
      {
        "eid": "E21",
        "score": 0.601220428943634,
        "route": "kg_semantic_name_only",
        "text": "Semantic: RHEUMATIC FEVER, HISTORY OF\n(SUI=S0399194)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S0399194"
        }
      },
      {
        "eid": "E22",
        "score": 0.601220428943634,
        "route": "kg_semantic_name_only",
        "text": "Semantic: History of - rheumatic fever\n(SUI=S7846030)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S7846030"
        }
      },
      {
        "eid": "E23",
        "score": 0.597686231136322,
        "route": "dense_def_faiss",
        "text": "Definition: arthritis caused by bacteria, rickettsiae, mycoplasmas, viruses, fungi, or parasites; bacterial arthritis is frequently caused by Staphylococcus aureus, Streptococcus pneumoniae, Haemophilus influenzae, Borrelia, and Neisseria gonorrhoeae; viral arthritis is less common than bacterial arthritis and may be a manifestation of such viral diseases as mumps, rubella, hepatitis, etc.\n(ATUI=AT51220761 SAB=CSP)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT51220761",
          "SAB": "CSP"
        },
        "ATUI": "AT51220761",
        "SAB": "CSP"
      },
      {
        "eid": "E24",
        "score": 0.5928216576576233,
        "route": "dense_def_faiss",
        "text": "Definition: A species of facultatively anaerobic, Gram positive, cocci shaped bacteria in the phylum Firmicutes. This species is beta hemolytic, Lancefield group A, pyrrolidonylarylamidase, and arginine deaminase positive and catalase negative. It can ferment salicin, rhamnose, and trehalose but not sorbitol, or ribose. S. pyogenes is found on normal human skin but can act as a pathogen causing streptococcal pharyngitis, acute rheumatic fever, scarlet fever, skin infections, acute glomerulonephritis and necrotizing fasciitis.\n(ATUI=AT198085651 SAB=NCI)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT198085651",
          "SAB": "NCI"
        },
        "ATUI": "AT198085651",
        "SAB": "NCI"
      },
      {
        "eid": "E25",
        "score": 0.5895950794219971,
        "route": "kg_sui_concept_def",
        "text": "Definition: \"A febrile disease occurring as a delayed sequela of infections with STREPTOCOCCUS PYOGENES. It is characterized by multiple focal inflammatory lesions of the connective tissue structures, such as the heart, blood vessels, and joints (POLYARTHRITIS) and brain, and by the presence of ASCHOFF BODIES in the myocardium and skin.\"\n(ATUI=\"AT115063354\" SAB=\"MSH\")",
        "trace": {
          "route": "kg_sui_concept_def",
          "seed_sui": "S0082534",
          "sem_vid": "S0082534",
          "concept_vid": "C0035436",
          "def_vid": "AT115063354",
          "atui": "AT115063354",
          "sab": "MSH"
        },
        "ATUI": "AT115063354",
        "SAB": "MSH"
      },
      {
        "eid": "E26",
        "score": 0.5809773802757263,
        "route": "dense_def_faiss",
        "text": "Definition: Infections in the inner or external eye caused by microorganisms belonging to several families of bacteria. Some of the more common genera found are Haemophilus, Neisseria, Staphylococcus, Streptococcus, and Chlamydia.\n(ATUI=AT38153569 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT38153569",
          "SAB": "MSH"
        },
        "ATUI": "AT38153569",
        "SAB": "MSH"
      },
      {
        "eid": "E27",
        "score": 0.5708827376365662,
        "route": "dense_def_faiss",
        "text": "Definition: Immunoassay reagents used in rapid tests to detect Streptococcus, a species of coccoid bacteria, in a patient's sample. Streptococcus is one of the most important bacterial pathogens of humans causing a wide range of suppurative diseases (e.g., pharyngitis, otitis media, sinusitis, endocarditis) and nonsuppurative sequelae (e.g., acute rheumatic fever).\n(ATUI=AT271213735 SAB=UMD)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT271213735",
          "SAB": "UMD"
        },
        "ATUI": "AT271213735",
        "SAB": "UMD"
      },
      {
        "eid": "E28",
        "score": 0.5694339871406555,
        "route": "kg_semantic_name_only",
        "text": "Semantic: History of rheumatic fever (situation)\n(SUI=S14182288)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S14182288"
        }
      },
      {
        "eid": "E29",
        "score": 0.5631932616233826,
        "route": "kg_semantic_name_only",
        "text": "Semantic: Rheumatic fever in the past 12Mo\n(SUI=S13459955)",
        "trace": {
          "route": "kg_semantic_name_only",
          "seed_sui": "S13459955"
        }
      },
      {
        "eid": "E30",
        "score": 0.5617130994796753,
        "route": "dense_def_faiss",
        "text": "Definition: A febrile disease occurring as a delayed sequela of infections with STREPTOCOCCUS PYOGENES. It is characterized by multiple focal inflammatory lesions of the connective tissue structures, such as the heart, blood vessels, and joints (POLYARTHRITIS) and brain, and by the presence of ASCHOFF BODIES in the myocardium and skin.\n(ATUI=AT115063354 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT115063354",
          "SAB": "MSH"
        },
        "ATUI": "AT115063354",
        "SAB": "MSH"
      },
      {
        "eid": "E31",
        "score": 0.5611706972122192,
        "route": "dense_def_faiss",
        "text": "Definition: A species of facultatively anaerobic, Gram positive, rod or cocci shaped bacteria assigned to the phylum Firmicutes. This species is nonmotile, non spore forming, catalase and oxidase negative, and produces pyrrolidonyl arylamidase. G. adiancens is a pathogen which can be isolated from the throat, blood or urine and most commonly causes endocarditis.\n(ATUI=AT198048670 SAB=NCI)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT198048670",
          "SAB": "NCI"
        },
        "ATUI": "AT198048670",
        "SAB": "NCI"
      },
      {
        "eid": "E32",
        "score": 0.5611612200737,
        "route": "dense_def_faiss",
        "text": "Definition: A species of gram-negative, aerobic, rod-shaped bacteria which is distinguished from other members of the genus KINGELLA by its beta hemolysis. It occurs normally in human mucous membranes of the upper respiratory tract, but can cause septic arthritis and endocarditis. (From Bergey's Manual of Determinative Bacteriology, 9th ed)\n(ATUI=AT38148059 SAB=MSH)",
        "trace": {
          "route": "dense_def_faiss",
          "ATUI": "AT38148059",
          "SAB": "MSH"
        },
        "ATUI": "AT38148059",
        "SAB": "MSH"
      }
    ]
  }